# Junções, integração e reprodutibilidade

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_03/03_juncoes_integracao_e_reprodutibilidade.ipynb)

O Notebook 02 produziu uma tabela intermediária com campos normalizados,
incertezas representadas e possíveis duplicatas sinalizadas. Este notebook
examina o que acontece quando essa tabela passa a se relacionar com outras:
uma junção pode confirmar vínculos, revelar lacunas ou criar casos indevidos.

### Como percorrer este notebook

A primeira célula de código apenas prepara o ambiente: no Colab, ela localiza
o repositório e instala o que estiver ausente; em uma execução local, apenas
informa que nenhuma clonagem é necessária. **Ela ainda não transforma os
dados da pesquisa.** Execute-a uma vez e, depois, acompanhe cada exemplo pela
sequência: pergunta → explicação → código → interpretação da saída.

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_03'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = [('openpyxl', 'openpyxl>=3.1,<4'), ('pypdf', 'pypdf>=5,<6'), ('PIL', 'Pillow>=11,<12')]
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

Ambiente preparado em: /content/disciplina_computacao_aplicada_humanidades_digitais/unidade_03


## 1. Chaves e cardinalidade

Antes de juntar, declare a unidade de cada tabela e a cardinalidade esperada:
1:1, 1:N ou N:N. Uma chave duplicada pode multiplicar linhas e fabricar peso
analítico. `validate` testa a hipótese de cardinalidade; `indicator` mostra a
cobertura da correspondência.

![Três painéis representam relações um para um, um para muitos e muitos para muitos, mostrando por que a cardinalidade precisa ser declarada e testada antes da junção.](https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_03/imagens/03_cardinalidades.svg?raw=1)

Cardinalidade descreve quantas linhas de um lado podem corresponder a cada
linha do outro. Ela deve ser formulada, confrontada com os dados e testada no
código antes da junção.

### Primeiro exemplo: acrescentar o nome oficial do município

Queremos enriquecer o catálogo de documentos com informações da tabela de
municípios. Antes da junção, o código abre os dois arquivos e registra o que
cada um representa:

| Objeto no código | Unidade de cada linha | Chave usada |
|---|---|---|
| `catalogo` | um documento | `codigo_municipio` |
| `municipios` | um município | `codigo_municipio` |

Como vários documentos podem pertencer ao mesmo município, mas cada código
deve identificar somente um município na tabela de referência, a relação
esperada é **muitos para um** (`many_to_one`). A célula seguinte somente lê os
arquivos e apresenta suas dimensões; ainda não os combina.

In [ ]:
import pandas as pd

catalogo = pd.read_csv(
    "dados/intermediarios/catalogo_normalizado.csv",
    dtype={"codigo_municipio": "string"},
)
municipios = pd.read_csv(
    "dados/brutos/extrato_codigos_municipios_ibge.csv",
    dtype={"codigo_municipio": "string"},
)

pd.DataFrame({
    "tabela": ["catalogo", "municipios"],
    "linhas": [len(catalogo), len(municipios)],
    "colunas": [len(catalogo.columns), len(municipios.columns)],
    "unidade_da_linha": ["documento", "município"],
})

,tabela,linhas,colunas,unidade_da_linha
0,catalogo,8,18,documento
1,municipios,4,3,município


Com as tabelas carregadas, podemos realizar a junção. Leia os argumentos do
código como uma declaração metodológica:

- `on="codigo_municipio"`: procura códigos iguais nas duas tabelas;
- `how="left"`: mantém todos os documentos, mesmo quando não encontra um
  município correspondente;
- `validate="many_to_one"`: exige que cada código da tabela municipal apareça
  no máximo uma vez;
- `indicator=True`: cria `_merge`, uma coluna que informa se cada linha foi
  encontrada nas duas tabelas (`both`) ou somente em uma delas.

Ao final, a contagem de `_merge` mostra a cobertura da junção. O resultado
desejado é que os oito documentos estejam em `both`; qualquer `left_only`
exigiria investigação.

In [ ]:
integrada = catalogo.merge(
    municipios,
    on="codigo_municipio",
    how="left",
    validate="many_to_one",
    indicator=True,
)

integrada["_merge"].value_counts()

,count
_merge,
both,8
left_only,0
right_only,0


A junção foi executada com uma cardinalidade declarada e um indicador de
correspondência. O resultado técnico precisa agora ser auditado: preservar o
número de linhas não garante que todas as chaves tenham sido encontradas.

### Auditoria da junção

`left_only` não deve ser descartado automaticamente: pode indicar código
inválido, cobertura incompleta da tabela de referência ou mudança temporal.
Compare contagens antes e depois e examine chaves sem correspondência.

A célula seguinte reúne cinco verificações em um pequeno relatório:

| Campo do relatório | O que permite verificar |
|---|---|
| `linhas_antes` e `linhas_depois` | se a junção multiplicou ou removeu documentos |
| `sem_correspondencia` | quantos documentos não encontraram município |
| `ids_unicos_antes` e `ids_unicos_depois` | se a quantidade de documentos distintos foi preservada |

`len` conta linhas; `nunique` conta identificadores distintos; e
`eq("left_only").sum()` conta as linhas marcadas como sem correspondência.

In [ ]:
auditoria_juncao = {
    "linhas_antes": len(catalogo),
    "linhas_depois": len(integrada),
    "sem_correspondencia": int(integrada["_merge"].eq("left_only").sum()),
    "ids_unicos_antes": int(catalogo["id_documento"].nunique()),
    "ids_unicos_depois": int(integrada["id_documento"].nunique()),
}
pd.Series(auditoria_juncao)

,0
linhas_antes,8
linhas_depois,8
sem_correspondencia,0
ids_unicos_antes,8
ids_unicos_depois,8


A auditoria mostrou como avaliar uma relação entre duas tabelas. Uma base de
pesquisa, porém, costuma reunir relações diferentes; textos, temas e
indicadores não compartilham necessariamente a mesma cardinalidade.

## 2. Integrar textos, metadados e indicadores

**Integrar** significa relacionar informações que descrevem a mesma unidade
sem confundir unidades diferentes. Nesta base, há quatro tipos de registro:

| Informação | Unidade de cada linha ou item | Relação com um documento |
|---|---|---|
| catálogo normalizado | um documento | uma linha por documento |
| arquivo textual | o texto disponível de um documento | zero ou um texto nesta versão |
| atribuição temática | uma relação documento–tema | um documento pode ter vários temas |
| indicador | uma relação documento–tema–período | um documento pode ter vários indicadores |

A distinção é importante porque “vários temas” não significa “vários
documentos”. Se repetíssemos a linha inteira de D001 para registrar educação e
progresso, também repetiríamos título, município e número de palavras. Uma
soma posterior poderia contar o mesmo documento duas vezes.

![A tabela de documentos conecta-se por chaves a textos, relações documento-tema, municípios e indicadores, com marcações das cardinalidades esperadas.](https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_03/imagens/03_modelo_relacional_base.svg?raw=1)

O desenho não exige colocar tudo em um único arquivo. Ele mantém uma tabela
central de documentos e tabelas auxiliares ligadas pelo identificador
`id_documento`. A seção seguirá quatro passos:

1. abrir os metadados que indicam arquivos textuais e temas;
2. criar separadamente a tabela de textos e a tabela documento–tema;
3. ligar os textos disponíveis ao catálogo sem remover documentos sem texto;
4. reorganizar os indicadores e exportar os três produtos derivados.

### 2.1 Ler o registro que orienta a integração

O JSON contém um objeto por documento. D001, por exemplo, aponta para
`D001.txt` e possui dois temas. O código abaixo apenas abre e apresenta esses
objetos; ainda não realiza nenhuma junção.

Leia a célula em quatro movimentos:

1. `Path(...)` representa o caminho do arquivo;
2. `read_text(...)` lê seu conteúdo como texto;
3. `json.loads(...)` interpreta esse texto como objetos Python;
4. `pd.DataFrame(...)` apenas os apresenta em forma de tabela para inspeção.

A variável `metadados` continua preservando a estrutura do JSON. A conversão
final em `DataFrame` serve somente para tornar a saída mais fácil de ler neste
momento.

In [ ]:
import json
from pathlib import Path

caminho_metadados = Path("dados/brutos/metadados.json")
texto_json = caminho_metadados.read_text(encoding="utf-8")
metadados = json.loads(texto_json)

pd.DataFrame(metadados)

,id_documento,arquivo_texto,temas
0,D001,D001.txt,"[educação, progresso]"
1,D002,D002.txt,[trabalho]
2,D003,None,[educação]


### 2.2 Criar as tabelas de temas e textos

Nesta etapa, aprenderemos que os dados podem ser mantidos em **tabelas
separadas conforme o que cada linha representa**. A separação não rompe a
relação entre eles: todas as tabelas conservam `id_documento` como chave
comum e podem ser reunidas posteriormente, quando a pergunta de pesquisa
exigir.

| Tabela | O que uma linha representa |
|---|---|
| tabela central de documentos | um documento |
| tabela de textos | o texto disponível de um documento |
| tabela documento–tema | a associação entre um documento e um tema |

Esse arranjo evita repetir indevidamente a linha completa de um documento.
D001, por exemplo, é um documento associado a dois temas — não dois
documentos diferentes.

Para transformar a lista de temas em tabela, precisamos produzir **uma linha
para cada relação**. O primeiro `for` percorre documentos; o segundo percorre
os temas daquele documento. Assim, D001 produzirá duas linhas, sem duplicar a
linha do catálogo.

| `id_documento` | `tema` |
|---|---|
| D001 | educação |
| D001 | progresso |

O mesmo bloco cria uma segunda tabela com apenas os textos disponíveis. A
condição `if arquivo_texto is not None` evita tentar abrir um arquivo quando o
JSON informa que ele não existe nesta versão. Ausência de texto não significa
ausência do documento no catálogo.

In [ ]:
linhas_temas = []
linhas_textos = []

# Primeiro, percorremos um documento por vez.
for item in metadados:
    id_documento = item["id_documento"]

    # Cada tema do documento se torna uma relação em uma nova linha.
    for tema in item["temas"]:
        linhas_temas.append({
            "id_documento": id_documento,
            "tema": tema,
        })

    # O texto só é lido quando o JSON informa um nome de arquivo.
    arquivo_texto = item["arquivo_texto"]
    if arquivo_texto is not None:
        caminho = Path("dados/brutos") / arquivo_texto
        linhas_textos.append({
            "id_documento": id_documento,
            "texto": caminho.read_text(encoding="utf-8"),
        })

tabela_temas = pd.DataFrame(linhas_temas)
tabela_textos = pd.DataFrame(linhas_textos)

print("Relações documento–tema:", len(tabela_temas))
print("Textos disponíveis:", len(tabela_textos))
tabela_temas

Relações documento–tema: 4
Textos disponíveis: 2


,id_documento,tema
0,D001,educação
1,D001,progresso
2,D002,trabalho
3,D003,educação


As duas tabelas agora preservam unidades distintas: uma linha da
`tabela_temas` é uma atribuição temática; uma linha da `tabela_textos` é um
texto disponível. Somente a tabela de textos pode ser ligada diretamente à
tabela central sem multiplicar documentos, pois há no máximo um texto por ID
nesta versão.

### 2.3 Ligar os textos sem excluir documentos

A junção abaixo parte de `integrada`, que possui oito documentos, e procura um
texto com o mesmo `id_documento`:

- `how="left"` preserva todos os documentos da tabela da esquerda;
- `on="id_documento"` declara a chave compartilhada;
- `validate="one_to_one"` interrompe a execução se um ID aparecer repetido em
  qualquer uma das duas tabelas.

O resultado continua com oito documentos. D001 e D002 recebem texto; os demais
permanecem com valor ausente na coluna `texto`. Essa ausência é informação
sobre a cobertura textual da versão, não motivo para apagar a linha.

In [ ]:
base_documentos = integrada.merge(
    tabela_textos,
    on="id_documento",
    how="left",
    validate="one_to_one",
)

print("Documentos antes da junção textual:", len(integrada))
print("Documentos depois da junção textual:", len(base_documentos))
base_documentos[["id_documento", "titulo", "texto"]].head(4)

Documentos antes da junção textual: 8
Documentos depois da junção textual: 8


,id_documento,titulo,texto
0,D001,Jornal Aurora,A escola noturna foi apresentada como sinal de...
1,D002,gazeta popular,O jornal discutiu jornadas de trabalho e instr...
2,D003,Correio do Vale,NaN
3,D004,JORNAL AURORA,NaN


Textos e temas já foram separados segundo suas cardinalidades, e a junção
textual preservou a quantidade de documentos. Falta aplicar a mesma lógica
aos indicadores.

### 2.4 Transformar indicadores largos em relações explícitas

Na tabela recebida, nomes como `educacao_1890` misturam duas informações na
mesma coluna: tema e período. `melt` mantém `id_documento` e transforma as
demais colunas em pares `tema_periodo`–`ocorrencias`.

Em seguida, `rsplit("_", n=1)` divide cada nome pelo último sublinhado:
`educacao_1890` torna-se tema `educacao` e período `1890`. Essa regra depende
de uma convenção documentada: todo nome precisa terminar em sublinhado e quatro
dígitos de ano.

In [ ]:
largo = pd.read_csv("dados/brutos/indicadores_largos.csv")
indicadores = largo.melt(
    id_vars="id_documento",
    var_name="tema_periodo",
    value_name="ocorrencias",
)

tema_e_periodo = indicadores["tema_periodo"].str.rsplit(
    "_", n=1, expand=True
)
tema_e_periodo.columns = ["tema", "periodo"]

indicadores[["tema", "periodo"]] = tema_e_periodo
indicadores = indicadores[
    ["id_documento", "tema", "periodo", "ocorrencias"]
]
indicadores.head(8)

,id_documento,tema,periodo,ocorrencias
0,D001,educacao,1890,3
1,D002,educacao,1890,0
2,D003,educacao,1890,2
3,D001,trabalho,1890,1
4,D002,trabalho,1890,4
5,D003,trabalho,1890,0
6,D001,educacao,1900,2
7,D002,educacao,1900,1


A transformação não calculou novos indicadores nem resumiu valores; apenas
mudou sua organização. Agora cada linha declara diretamente sua unidade:
documento–tema–período. Com as três tabelas prontas, podemos exportá-las.

### 2.5 Exportar produtos derivados

Cada comando `to_csv` grava uma tabela com função própria. A coluna `_merge`
foi útil para auditar a junção municipal, mas não pertence ao produto final e
é retirada da tabela de documentos.

In [ ]:
base_documentos.drop(columns="_merge").to_csv(
    "dados/derivados/documentos_processaveis.csv", index=False
)
tabela_temas.to_csv(
    "dados/derivados/documentos_temas.csv", index=False
)
indicadores.to_csv(
    "dados/derivados/indicadores_longos.csv", index=False
)

pd.Series({
    "documentos": len(base_documentos),
    "relações documento–tema": len(tabela_temas),
    "relações documento–tema–período": len(indicadores),
})

,0
documentos,8
relações documento–tema,4
relações documento–tema–período,12


O modelo relacional preserva entidades e multiplicidades em tabelas ligadas.
Para que esse desenho seja reproduzível, também precisamos organizar arquivos,
ordem de execução, parâmetros e produtos derivados.

## 3. Organização e execução

Uma estrutura simples separa entrada, intermediários e derivados. Notebooks
numerados tornam a ordem visível, mas reprodutibilidade também requer ambiente,
parâmetros, versões e execução desde o início. Saídas derivadas devem poder ser
reconstruídas sem editar manualmente células intermediárias.

Antes de ler o código, traduza cada verificação para uma pergunta sobre a
pesquisa:

| Verificação no código | Pergunta em linguagem comum | Resultado esperado |
|---|---|---|
| IDs únicos | cada linha da tabela central representa um documento diferente? | `True` |
| nenhuma linha criada | as junções evitaram multiplicar documentos? | `True` |
| temas referenciam documentos | todo ID usado na tabela de temas existe na tabela central? | `True` |
| arquivos derivados | os três produtos planejados foram gravados? | `3` |

`is_unique` responde à primeira pergunta. A comparação de comprimentos testa
a segunda. `issubset` verifica se o conjunto de IDs dos temas está contido no
conjunto de documentos. Por fim, `glob("*.csv")` localiza os arquivos CSV na
pasta de derivados e `len` os conta.

In [ ]:
from pathlib import Path

verificacoes = {
    "ids_documentos_unicos": base_documentos["id_documento"].is_unique,
    "nenhuma_linha_criada_nas_juncoes": (
        len(base_documentos) == len(catalogo)
    ),
    "temas_referenciam_documentos": set(
        tabela_temas["id_documento"]
    ).issubset(set(base_documentos["id_documento"])),
    "arquivos_derivados": len(list(Path("dados/derivados").glob("*.csv"))),
}
pd.Series(verificacoes)

,0
ids_documentos_unicos,True
nenhuma_linha_criada_nas_juncoes,True
temas_referenciam_documentos,True
arquivos_derivados,3


As verificações anteriores testam se arquivos, chaves e relações foram
reconstruídos como esperado. O plano de integração transforma esses testes em
um desenho explícito que pode ser revisado antes de receber dados reais.

## U03-A04 — Atividade — plano de integração

Desenhe as tabelas, unidades, chaves, cardinalidades, campos compartilhados,
validações, tratamento de não correspondências e saídas. Indique como um
resultado será rastreado até a fonte.

**Plano:**

1. **Estrutura das Tabelas e Unidades de Análise:**
   * **Tabela Central (`documentos_processaveis.csv`):** Unidade = *um discurso de posse presidencial* (`id_doc`). Contém metadados primários (`id_doc`, `ano`, `presidente`, `partido`) e a transcrição integral limpa.
   * **Tabela Derivada de Trechos (`discursos_trechos.csv`):** Unidade = *um parágrafo contendo a palavra "liberdade"* (`trecho_id`). Contém a frase/parágrafo do contexto imediato, o indicador de ocorrência e os campos semânticos associados.
   * **Tabela de Descartes (`motivos_exclusao.csv`):** Unidade = *um documento/pronunciamento descartado*. Contém o `id_doc`, motivo da exclusão e justificativa.

2. **Chaves, Cardinalidade e Regras de Junção:**
   * **Relação Documentos → Trechos (`1:N` / `one_to_many`):** Cada discurso (`id_doc`) pode gerar múltiplos trechos contendo a palavra "liberdade" (`trecho_id`).
   * **Regra de Junção (`pd.merge`):** Junções do tipo `left` garantem que nenhum discurso formal seja suprimido silenciosamente.
   * **Validação de Cardinalidade (`validate="one_to_many"`):** Valida formalmente a expansão de 12 documentos em múltiplos parágrafos analisáveis.

3. **Estratégia de Validação e Testes de Integridade:**
   * **Teste de Integridade de Chaves:** Verificação de unicidade das chaves primárias (`is_unique`) em `id_doc` (na tabela central) e em `trecho_id` (na tabela derivada).
   * **Indicadores de Junção:** Uso do parâmetro `indicator=True` no Pandas para confirmar a consistência de correspondência total (`_merge == "both"`) entre metadados e trechos.
   * **Rastreabilidade de Referência:** Todo `trecho_id` preserva obrigatoriamente a chave estrangeira `id_doc`, permitindo recompor o discurso de origem na íntegra.

4. **Saídas Derivadas e Reprodutibilidade:**
   * Exportação dos arquivos consolidados para a camada `dados/derivados/` (`documentos_processaveis.csv` e `discursos_trechos.csv`).
   * Registro completo dos parâmetros, datas e etapas de execução no arquivo JSON de proveniência (`proveniencia_pipeline.json`), garantindo auditoria e reprodutibilidade de ponta a ponta.

O plano produzido na atividade reúne unidades, chaves, cardinalidades,
validações e rastreamento. A síntese destaca a consequência metodológica desse
desenho: integrar dados é afirmar identidades e relações entre entidades.

## Síntese

Junção é uma afirmação de identidade e relação, não mero encaixe de colunas.
A base processável pode ser plural: tabela de documentos, tabela de relações,
textos e indicadores ligados por chaves verificadas. Leve esse pacote para a
oficina, onde inventário, transformações, integrações, testes e documentação
serão avaliados como uma única entrega.